In [1]:
import mediapy as media
import numpy as np
import sys
import jax
import jax.numpy as jp
from datetime import datetime
from mujoco_playground._src import mjx_env
import mujoco
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from pathlib import Path
from mujoco import mjx
from ml_collections import config_dict
from mujoco_playground import wrapper
from mujoco_playground.config import manipulation_params
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
REPO = Path.cwd().parent          # training/ -> robotic-craftsman/
sys.path.insert(0, str(REPO))
from arm import ROOT, NPZ_PATH, XML_PATH, Calibration, HandPose, MjcArm, DepthScale, ArmIK, STServo, FingerBoard


CTRL_DT = 1/30
SIM_DT = 0.002 #timestep in sim
n_substeps = round(CTRL_DT / SIM_DT)
episode_length = 260
EMA_ALPHA = 0.8 #exponential moving average  - smoothing filter
noise = 0.01
tracking_sigma = 0.3 #eval of the copying score
max_tracking_error = 0.8 # maximum error allowance for target to arrive at
REWARD_WEIGHTS = {
    "tracking": 1.0,
    "pinch": 0.0,
    "action_rate": -0.01,
    "termination": -5.0
}

In [2]:
CLIP = ROOT / "refs" / "clip02.npz"
Q_REF = np.load(CLIP)["q_ref"]
Q_REF = Q_REF[2:]

In [ ]:
print((Q_REF[:, 5].min()),(Q_REF[:, 5].max())))

In [5]:
def default_config():
       return config_dict.create(ctrl_dt=CTRL_DT, sim_dt=SIM_DT, episode_length=episode_length)
   

class HandTraining(mjx_env.MjxEnv):
    
    idx_list: list

    
    def __init__(self, config):
        self.POLICY_ACTS = ["a_elbow_pitch", "a_forearm_roll", "a_wrist_tilt", "a_index_mcp", "a_thumb_pip", "a_thumb_mcp"]
        self.sim = MjcArm()
        self.Q_REF = jp.array(Q_REF)

        self.sim.reset()
        self.init_data = mjx.put_data(self.sim.model, self.sim.data)
        super().__init__(config)  
        self.mx = mjx.put_model(self.sim.model)
        self.sim.reset()
        self.qpos_adr = []
        self.qvel_adr = []
        self.idx_list = []
        for name in self.POLICY_ACTS:
            self.idx_list.append(self.sim.actuator_id(name))
        self.LOWERS, self.UPPERS = self.sim.model.actuator_ctrlrange[jp.array(self.idx_list)].T
        for idx in self.idx_list:
            init_pos = self.sim.model.actuator_trnid[idx, 0]
            self.qpos_adr.append(self.sim.model.jnt_qposadr[init_pos])
            self.qvel_adr.append(self.sim.model.jnt_dofadr[init_pos])
        self.qpos_adr = np.array(self.qpos_adr)
        self.qvel_adr = np.array(self.qvel_adr)
        self.idx_list = np.array(self.idx_list)
        self.action_scale()

    @property
    def action_size(self):
        return len(self.POLICY_ACTS)

    @property
    def mj_model(self):
        return self.sim.model

    @property
    def mjx_model(self):
        return self.mx

    @property
    def xml_path(self):
        return XML_PATH

    def _get_obs(self, dx, info):
        return jp.concatenate([
            dx.qpos[self.qpos_adr],
            dx.qvel[self.qvel_adr],
            info["last_action"],
            self.Q_REF[info["t"] + 1] - dx.qpos[self.qpos_adr],
            ])

    def action_scale(self):
        q = np.asarray(self.Q_REF)
        biggest_jump = []
        for i in range(q.shape[1]):
            fr_diff = []
            for fr in range(len(q) - 1):
                fr_diff.append(np.abs(q[fr, i] - q[fr + 1, i]))
            biggest_jump.append(max(fr_diff))
        self.ACTION_SCALE = jp.array(np.maximum(np.array(biggest_jump) * 1.5, 0.05 * (self.UPPERS - self.LOWERS)))
        return dict(zip(self.POLICY_ACTS, np.asarray(self.ACTION_SCALE).round(3)))
    
    def reset(self, rng):
        rng, key = jax.random.split(rng)
        dx = self.init_data
        t0 = jax.random.randint(key, (), 0, len(self.Q_REF) - self._config.episode_length)
        q0 = self.Q_REF[t0]
        dx = dx.replace(qpos=dx.qpos.at[self.qpos_adr].set(q0))
        dx = dx.replace(ctrl=dx.ctrl.at[self.idx_list].set(q0))
        info = {
            "rng": rng,
            "t": t0,
            "step": jp.array(0),
            "last_action": jp.zeros(self.action_size)
        }
        dx = mjx.forward(self.mx, dx)
        obs = self._get_obs(dx, info)
        return mjx_env.State(dx, obs, jp.zeros(()), jp.zeros(()), {"tracking": jp.zeros(())}, info)
    
    def step(self, state, action):
        dx = state.data
        info = state.info
        t = info["t"]
        target = jp.clip(self.Q_REF[t + 1] + action * self.ACTION_SCALE, self.LOWERS, self.UPPERS)
        dx = dx.replace(ctrl=dx.ctrl.at[self.idx_list].set(target))
        t += 1
        for _ in range(n_substeps):
            dx = mjx.step(self.mx, dx)

        err = dx.qpos[self.qpos_adr] - self.Q_REF[t]
        tracking = jp.exp(-jp.sum(err**2) / tracking_sigma**2)
        metrics = {"tracking": tracking}
        action_rate = jp.sum((action - info['last_action'])**2)
        done = ((jp.linalg.norm(err) > max_tracking_error) | jp.isnan(dx.qpos).any()).astype(jp.float32)
        reward = REWARD_WEIGHTS["tracking"] * tracking + REWARD_WEIGHTS["action_rate"] * action_rate
        reward = reward + REWARD_WEIGHTS["termination"] * done
        info = {**info, "t": t, "step": info["step"] + 1, "last_action": action}
        obs = self._get_obs(dx, info)

        return state.replace(data=dx, obs=obs, reward=reward, done=done, metrics=metrics, info=info)

            

In [ ]:
env = HandTraining(default_config())
state = jax.jit(env.reset)(jax.random.PRNGKey(0))
state2 = jax.jit(env.step)(state, jp.zeros(env.action_size))

x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]

def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, metrics["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

In [ ]:
step = jax.jit(env.step)
print("err at reset:", state.data.qpos[env.qpos_adr] - env.Q_REF[state.info["t"]])
s = state
for k in range(10):
    s = step(s, jp.zeros(env.action_size))
    print(k, np.round(s.data.qpos[env.qpos_adr] - env.Q_REF[s.info["t"]], 3), float(s.metrics["tracking"]))

err at reset: [0. 0. 0. 0. 0. 0.]


KeyboardInterrupt: 

In [ ]:

make_inference_fn, params, metrics = ppo.train(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    num_timesteps=100_000,
    num_evals=5,                 # how many times progress() fires
    episode_length=episode_length,
    num_envs=128,
    batch_size=64,
    num_minibatches=8,           # batch_size * num_minibatches must divide by num_envs
    unroll_length=10,
    num_updates_per_batch=4,
    learning_rate=3e-4,
    entropy_cost=1e-2,
    discounting=0.97,
    normalize_observations=True,
    progress_fn=progress(),
)


NameError: name 'wrapper' is not defined

In [38]:
#test cell
print("scales:", env.action_scale())
print("obs shape:", state.obs.shape)
print("t:", state.info["t"], "->", state2.info["t"])
print("tracking:", state2.metrics["tracking"], "done:", state2.done, "reward:", state2.reward)
print("any NaN:", jp.isnan(state2.obs).any())
print(np.round(np.diff(np.asarray(env.Q_REF[:12, 1])), 3))

scales: {'a_elbow_pitch': np.float32(0.157), 'a_forearm_roll': np.float32(0.515), 'a_wrist_tilt': np.float32(0.27), 'a_index_mcp': np.float32(0.093), 'a_thumb_pip': np.float32(0.078), 'a_thumb_mcp': np.float32(0.068)}
obs shape: (24,)
t: 0 -> 1
tracking: 0.24879944 done: 0.0 reward: 0.24879944
any NaN: False
[ 0.343 -0.091 -0.055 -0.1   -0.013 -0.099 -0.035  0.078 -0.036  0.05
 -0.022]


In [11]:
mx = mjx.put_model(sim.model)
dx = mjx.put_data(sim.model, sim.data)
dx = mjx.step(mx, dx)

mjx.get_data(sim.model, dx)
sid = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "index_touch")
adr = sim.model.sensor_adr[sid]
dx.sensordata[adr]

step_fn = jax.jit(mjx.step)       # mesh centre vs body origin
adr = sim.model.jnt_qposadr[sim.model.joint("needle_free").id]
sim.data.qpos[adr:adr + 3] = [1,1,1]

sim.set_ctrl("a_index_mcp",0.55)
sim.set_ctrl("a_thumb_pip", -0.05)
sim.set_ctrl("a_thumb_mcp", -0.0233)

mx = mjx.put_model(sim.model)
dx = mjx.put_data(sim.model, sim.data)

for _ in range (1500):
    dx = step_fn(mx, dx)

if np.isnan(dx.qpos).any():
    raise ValueError("NaN in qpos")

sid = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "index_touch")
sth = mujoco.mj_name2id(sim.model, mujoco.mjtObj.mjOBJ_SENSOR, "thumb_touch")
adr_i = sim.model.sensor_adr[sid]
adr_t = sim.model.sensor_adr[sth]
print(dx.sensordata[adr_i], dx.sensordata[adr_t])

42.66696 42.66696
